# Pathadol Neuro-Symbolic Brain — 1-Click Auto Trainer (Colab L4 GPU / T4 / CPU)

**Just click Play once on the cell below** (or `Runtime -> Run all`). You do **not** need to upload any scripts or `.jsonl` datasets:

1. Automatically fetches the latest [`pathadol_brain_train.py`](https://github.com/5ivesaw/pathadol/blob/main/colab/pathadol_brain_train.py) from GitHub.
2. Synthesizes teacher warm-start rollouts and trains [`PathadolBrainNet`](https://github.com/5ivesaw/pathadol/blob/main/colab/pathadol_brain_train.py) via Masked PPO self-play inside Pathadol's vectorized 4-team Bedwars macro simulator (~3.93M simulated decisions by default).
3. Exports `out/active.pbrain` (~450 KB pure-Java format), checks exact PyTorch vs Java forward-pass parity, and downloads `active.pbrain` automatically.

> *(Optional)* If you ever record real match `.jsonl` logs via `#brain record start` and drop them into Colab's `brain_data/` folder before running, Step 1 will automatically use them instead of synthetic teacher rollouts.

After downloading **`active.pbrain`**, place it at `.minecraft/pathadol/brain/active.pbrain` and run **`#brain reload`** then **`#brain start`** (or **`#bws auto`**) in-game.

In [ ]:
#@title 1-Click Run All: Auto-Download Trainer, Train Bedwars Brain & Download active.pbrain
BC_EPOCHS = 10 #@param {type:"integer"}
PPO_UPDATES = 120 #@param {type:"integer"}
NUM_ENVS = 512 #@param {type:"integer"}

import os
import shutil
import subprocess
import sys
import urllib.request
import torch

TRAINER_URL = "https://raw.githubusercontent.com/5ivesaw/pathadol/main/colab/pathadol_brain_train.py"
os.makedirs("brain_data", exist_ok=True)
os.makedirs("out", exist_ok=True)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    print("Fetching latest pathadol_brain_train.py from GitHub...")
    urllib.request.urlretrieve(TRAINER_URL, "pathadol_brain_train.py")
except Exception as exc:
    if os.path.exists("pathadol/colab/pathadol_brain_train.py"):
        shutil.copy2("pathadol/colab/pathadol_brain_train.py", "pathadol_brain_train.py")
    elif not os.path.exists("pathadol_brain_train.py"):
        raise RuntimeError(f"Failed to fetch pathadol_brain_train.py: {exc}") from exc

cmd = [
    sys.executable,
    "pathadol_brain_train.py",
    "--data-glob", "brain_data/*.jsonl",
    "--out", "out/active.pbrain",
    "--bc-epochs", str(BC_EPOCHS),
    "--ppo-updates", str(PPO_UPDATES),
    "--num-envs", str(NUM_ENVS),
]
print("Starting training:", " ".join(cmd))
subprocess.run(cmd, check=True)

try:
    from google.colab import files
    files.download("out/active.pbrain")
    print("Downloaded out/active.pbrain -> place at .minecraft/pathadol/brain/active.pbrain")
except ImportError:
    print("Saved trained brain to:", os.path.abspath("out/active.pbrain"))
